# Introducción a Keras

In [ ]:
# Compatibilidad de entorno y datos locales
import sys, types
from pathlib import Path
import numpy as np
import pandas as pd

# Compatibilidad para sklearn.datasets.samples_generator
try:
    import sklearn.datasets as _ds
    _mod = types.ModuleType("sklearn.datasets.samples_generator")
    _mod.make_blobs = _ds.make_blobs
    _mod.make_circles = _ds.make_circles
    sys.modules.setdefault("sklearn.datasets.samples_generator", _mod)
except Exception:
    pass

# Compatibilidad para keras.utils.np_utils
try:
    from tensorflow.keras.utils import to_categorical as _to_categorical
    _km = types.ModuleType("keras.utils.np_utils")
    _km.to_categorical = _to_categorical
    sys.modules.setdefault("keras.utils.np_utils", _km)
except Exception:
    pass

# Compatibilidad para load_boston
try:
    from sklearn import datasets as _skd
    if not hasattr(_skd, "load_boston"):
        from sklearn.datasets import load_diabetes
        from sklearn.utils import Bunch
        def _load_boston_fallback():
            d = load_diabetes()
            return Bunch(data=d.data, target=d.target, feature_names=d.feature_names, DESCR="Fallback load_boston")
        _skd.load_boston = _load_boston_fallback
except Exception:
    pass


## Importación de los módulos necesarios

In [ ]:
%matplotlib inline 
import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn import preprocessing 
from sklearn.model_selection import train_test_split

In [ ]:
digits = load_digits() ## importación del dataset

In [ ]:
## mostramos de un núemro random
sample_index = 45
plt.figure(figsize = (3, 3))
plt.imshow(digits.images[sample_index], cmap = plt.cm.gray_r, interpolation = 'nearest')
plt.title("label: %d" % digits.target[sample_index]);

## Preprocesado de los datos

In [ ]:
## Hacemos un preprocesado de los datos
digits.data_nor = preprocessing.scale(digits.data)
X_train_nor, X_test_nor, y_train, y_test = train_test_split(digits.data_nor, digits.target, test_size = 0.2)

Verificación de la buena importación de los datos

In [ ]:
print(X_train_nor.shape, y_train.shape)
print(X_test_nor.shape, y_test.shape)

# Red neuronal feed-forward con Keras

La primera parte es hacer el to categorical para la buena clasificación con las redes

In [ ]:
from tensorflow.keras.utils import to_categorical

Y_train = to_categorical(y_train);
Y_train[:3]

### Definición de la arquitectura de la celda

In [ ]:
from tensorflow.keras.models import Sequential

Con Secuential es una red que se propaga hacia adelante añadiendo progresivamente capas
- Dense Permite defenor las preactivaciones
- Activation: Par definir las funciones de activación

In [ ]:
from tensorflow.keras.layers import Dense, Activation
## dimension del modelo
N = X_train_nor.shape[1]  # dimension en entrada de la red
N

In [ ]:
H = 128 # dimension de la capa escondida
K = 10  # numero de classes en la salida (cantidad de targets)
 ## Arquitectura
model = Sequential()
model.add(Dense(H, input_dim=N))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1]))
model.add(Activation("softmax"))

model.summary() ## para ver qué show

La dimesniones en salida de cada capa son de lña forma (None,...). Cuando la red sea ajsutada **.fit** sobre un set (normalmente un train) la primera dimensión indicará la talla de este set

In [ ]:
# para ver la red
from tensorflow.keras.utils import plot_model
#plot_model(model, to_file='model.png') 

### Pesos de la red

**Primera capa** Dimension espetrada para la tabla de pesos : **64 x 128** donde 64 es el número de features y 128 es el número de neuronas de la primera capa.

**Capa de salida.** Dimension esperada : **128 x 10** donde 128 es el número d enetradas de la capa de salida y 10 es el número de neuronas de la capa de de salida.

In [ ]:
# se pueden ver los parámetros de l primera capa con:
my_first_layer = model.layers[0]

## Vamos a verificar que esté bien inicializada la capa
print("Dimension de la tabla de pesos para la primera capa : ", model.get_weights()[0].shape)
print("Número de pesos de la primera capa : ", model.get_weights()[1].shape)
print("Dimension de la tabla de pesos para la primera capa : ", model.get_weights()[2].shape)
print("Número de pesos de la primera capa : ", model.get_weights()[3].shape)


## estos pesos son inicializados a cero
print(model.get_weights()[1])
print(model.get_weights()[3])

La fase de aprendizaje consiste a estimar $\theta$ (que contiene todos los parámetros de la red) con un método de optimisación de tipo descenso de gradiente estocástico,

Por ejemplo para, **SGD**, cada iteración efectua un paso de gradiente sobre un sub-set de datos :
$$ \theta_b  =   \theta_{b-1}  - \varepsilon   \frac 1{batch size} \sum_{i \in B}  \nabla_\theta \ell( f(X_i , \theta_{b-1}),Y_i)   
 $$

In [1]:
from tensorflow.keras.optimizers import SGD
sgd = SGD(learning_rate=0.1);  # lr == learning rate epsilon en la fórmula anterior

Using TensorFlow backend.


> Ajustemos una primera red multilayer perceptron con una capa escondida

In [ ]:
model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])

## vemos que usamos la perdida croos entropy que es la pérdida usada en SGD y la métrica accuracy que es la metrica seca (atinar o no)

En los argumentos de **.fit** usamos:
- epochs = número de veces que pasa el aprendizaje en el set (una meta iteración)
- batch_size = talla del sub set extraído para el cálculo del gradiente en cada operación.

In [ ]:
model.fit(X_train_nor, Y_train, epochs = 20, batch_size = 32) ;

In [ ]:
## ahopra los pesos se ajustaron y ya no son cero. Si segumis entrenando a la red se quedará en donde la dejamos....
print("Les poids ne sont plus à zéro : \n", model.get_weights()[1][:10])
model.fit(X_train_nor, Y_train, epochs = 20, batch_size = 32) ;

> Ensayo con una taza de aprendizaje de 0.001

In [ ]:
model = Sequential()
model.add(Dense(H, input_dim=N))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1]))
model.add(Activation("softmax"))

sgd = SGD(learning_rate= 0.001)
model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
model.fit(X_train_nor, Y_train, epochs = 20, batch_size = 32) ;

> Ensayo con una taza de aprendizaje de 0.001 y momentum SGD

In [ ]:
model = Sequential()
model.add(Dense(H, input_dim=N))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1]))
model.add(Activation("softmax"))

sgd = SGD(learning_rate= 0.001, momentum=0.9)
model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
model.fit(X_train_nor, Y_train, epochs = 20, batch_size = 32) ;

> Ensayo con una taza de aprendizaje de 0.1 y momentum SGD

In [ ]:
model = Sequential()
model.add(Dense(H, input_dim=N))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1]))
model.add(Activation("softmax"))

sgd = SGD(learning_rate=0.1)
model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
model.fit(X_train_nor, Y_train, epochs = 20, batch_size = 32) ;

#### La convergencia es más lenta para una taza de aprendizaje pequeño. Remarcamos que si usamos el Momentum-SGD se permite augmentar la rapidez de convergencia.

#### Intentaemos para varios valores del learning rate

In [ ]:
liste_lr = [1,10,100]
for lr in liste_lr :
    model = Sequential()
    model.add(Dense(H, input_dim=N))
    model.add(Activation("tanh"))
    model.add(Dense(Y_train.shape[1]))
    model.add(Activation("softmax"))

    sgd = SGD(learning_rate=float(lr))
    model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
    model.fit(X_train_nor, Y_train, epochs = 20, batch_size = 32) ;

#### Se ve que si el learning rate es elevado la perdida oscila y no parece converger

> Estimación del efrror de genaralización con el set test

In [ ]:
model = Sequential()
model.add(Dense(H, input_dim=N))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1]))
model.add(Activation("softmax"))

sgd = SGD(learning_rate= 0.01, momentum=0.9)
model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
model.fit(X_train_nor, Y_train, epochs = 20, batch_size = 32, verbose = 0) ;
y_predicted = np.argmax(model.predict(X_test_nor), axis=1)

In [ ]:
## ver mas o mneos cómo nos fue...
plt.figure(figsize = (12, 9))
for i in range(15) :
    plt.subplot(3, 5, i + 1)
    plt.imshow(X_test_nor[i].reshape(8, 8), cmap = plt.cm.gray_r, interpolation = 'nearest')
    plt.title("predicted label: %d\n true label: %d" % (y_predicted[i], y_test[i]))

> Calculo del accuracy medio sobre el set test

In [ ]:
accuracy = 0
for i in range(len(y_predicted)) :
    if y_predicted[i] == y_test[i] :
        accuracy += 1
accuracy /= len(y_predicted)
print('%0.2e'%accuracy)

Al final lo que se calcula es una probabilidad condicional con loq ue ya aprendió la red: 

In [ ]:
sample_idx = 2
plt.imshow(X_test_nor[sample_idx].reshape(8, 8), cmap = plt.cm.gray_r, interpolation = 'nearest');
plt.title("predicted label: %d\n true label: %d" % (y_predicted[sample_idx], y_test[sample_idx])) ;

probabilities = model.predict(X_test_nor)
print(probabilities[sample_idx]) # proba condicional con la función softmax a la salida de la red

La pérdida empírica del cross entropy vale:
\begin{eqnarray*}
\hat{ \mathcal R}_n (\theta)  &=&  \frac 1n  \sum_{i=1}^n  \ell( Y_i,  f(X_i , \theta)) \\ 
& =&   -    \frac 1n \sum_{i=1}^n  \sum_{k = 1}^K  \mathbb{1}_{Y_i =  k}  \log (f(X_i , \theta)_k)  
\end{eqnarray*}

Encontremos esto directamente con la **función de Python**

In [ ]:
Y_test = to_categorical(y_test)
scores = model.evaluate(X_test_nor, Y_test)
print("Riesgo asociado a la perdida empícra : ", np.array(scores)[0])
print("Accuracy : ", np.array(scores)[1])

### Impacto de la inicialización

Vamos ahora a estudiar el impacto de la inicialización del algoritmo SGD en su convergencia

En Keras, por defcto los pesos son inicializados como sigue con el inicializador **glorot_uniform** :

- Cada peso se obtiene de con un aley uniforme en $[-scale, scale]$
- La scale es elegida del orden de $\frac{1}{\sqrt{n_{in} + n_{out}}}$

Vamos a estudiar el impacto de la inicialización consdierando inicializadores gaussianos para varios valores de la varianza.

In [ ]:
from tensorflow.keras import initializers
mon_init1 = initializers.RandomNormal(mean=0.0, stddev=0.05)

> Definra una red con una capa Softmax en salida y dos capas escondidas tanh con repsctivamente H layers y K layers.

> Escribir el inicializador en argumento 'Dense()', esto da como ejemplo en la primera capa:  
`model.add(Dense(K,input_dim=N,kernel_initializer=mon_init1))`.

In [ ]:
model = Sequential()
model.add(Dense(Y_train.shape[1], input_dim=N, kernel_initializer = mon_init1))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1], kernel_initializer = mon_init1))
model.add(Activation("tanh"))
model.add(Dense(H))
model.add(Activation("softmax"))

In [ ]:
# Ajustamos la red (.fit) a los datos train
num_classes = Y_train.shape[1]
model = Sequential()
model.add(Dense(H, input_dim=N))
model.add(Activation('tanh'))
model.add(Dense(num_classes))
model.add(Activation('softmax'))

sgd = SGD(learning_rate=0.01, momentum=0.9)
model.compile(optimizer=sgd, loss='categorical_crossentropy', metrics=['accuracy'])

# history sirve para rescatar despu?s la evoluci?n de par?metros, como el loss
history = model.fit(X_train_nor, Y_train, epochs=100, batch_size=32, verbose=0)

plt.plot(history.history['loss'], label='stddev=0.05')
plt.xlabel('# epochs')
plt.ylabel('Training loss')
plt.ylim(0, 6)
plt.legend(loc='best')


> Hacer lo mismo con los inicializadores siguientes :    
> `mon_init2 = initializers.RandomNormal(mean=0.0,stddev=10)`   
> `mon_init3 = initializers.RandomNormal(mean=0.0,stddev=0.001)` 
> `mon_init4 = initializers.glorot_uniform()` 

> Sacar conclusiones

In [ ]:
mon_init2 = initializers.RandomNormal(mean=0.0,stddev=10) # mon_init2 con stddev=10

model = Sequential()
model.add(Dense(H, input_dim=N, kernel_initializer = mon_init2))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1], kernel_initializer = mon_init2))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1]))
model.add(Activation("softmax"))


sgd = SGD(learning_rate= 0.01, momentum=0.9)
model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
history = model.fit(X_train_nor, Y_train, epochs = 100, batch_size = 32, verbose = 0)

plt.plot(history.history['loss'],label = "stddev=10")
plt.xlabel('# epochs')
plt.ylabel('Training loss')
plt.ylim(0, 6)
plt.legend(loc='best');

In [ ]:
mon_init3 = initializers.RandomNormal(mean=0.0,stddev=0.001)

model = Sequential()
model.add(Dense(H, input_dim=N, kernel_initializer = mon_init3))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1], kernel_initializer = mon_init3))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1]))
model.add(Activation("softmax"))

sgd = SGD(learning_rate= 0.01, momentum=0.9)
model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
history = model.fit(X_train_nor, Y_train, epochs = 60, batch_size = 32, verbose = 0)

plt.plot(history.history['loss'],label = "stddev=0.001")
plt.xlabel('# epochs')
plt.ylabel('Training loss')
plt.ylim(0, 6)
plt.legend(loc='best');

In [ ]:
mon_init4 = initializers.glorot_uniform()

model = Sequential()
model.add(Dense(H, input_dim=N, kernel_initializer = mon_init4))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1], kernel_initializer = mon_init4))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1]))
model.add(Activation("softmax"))

sgd = SGD(learning_rate= 0.01, momentum=0.9)
model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
history = model.fit(X_train_nor, Y_train, epochs = 20, batch_size = 32, verbose = 0)

plt.plot(history.history['loss'], label = "glorot_uniform")
plt.xlabel('# epochs')
plt.ylabel('Training loss')
plt.ylim(0, 6)
plt.legend(loc = 'best');

## Hacer una comparación en las trayectorias de 

#### Saque conclusiones

In [ ]:
counter = 0

for init in [mon_init1, mon_init2, mon_init3, mon_init4] :
    counter += 1
    model = Sequential()
    model.add(Dense(H, input_dim=N, kernel_initializer = init))
    model.add(Activation("tanh"))
    model.add(Dense(num_classes, kernel_initializer=init))
    model.add(Activation("tanh"))
    model.add(Dense(num_classes))
    model.add(Activation("softmax"))

    sgd = SGD(learning_rate= 0.01, momentum=0.9)
    model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
    history = model.fit(X_train_nor, Y_train, epochs = 50, batch_size = 32, verbose = 0)

    label = "init" + str(counter)
    plt.plot(history.history['loss'],label = label)
    plt.xlabel('# epochs')
    plt.ylabel('Training loss')
    plt.legend(loc = 'best');

- Inicialización muy próxima acero : Gradientes muy débiles (pequeños) y a SGD le cuesta salir de esa zona
- Inicialización con amplitudes muy elevedas: Las caoas interedias tienengradientes que se anulan y es dificil salir de esa zona.

## Early Stopping

El argumento **validation_split = 0.1** en el método **.fit()** permite de evaluar la pérdida en el set d evalidación.

> Mostrar gráficamente que después de un cierto número de iteraciones (aquí entre 1000 y 2000). el riesgo (loss) y la precisión (accuracy) en el set de validación acaban por degradarse.

Este fenómeno ilustar el interés del método conocido como **Early stopping**


In [ ]:
liste_epochs=[20,40,60]

for epoch in liste_epochs :
    model = Sequential()
    model.add(Dense(H, input_dim = N))
    model.add(Activation("tanh"))
    model.add(Dense(Y_train.shape[1]))
    model.add(Activation("softmax"))
    sgd = SGD(learning_rate=0.1, momentum=0.9)
    model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'])
    history = model.fit(X_train_nor, Y_train, epochs = epoch, validation_split = 0.1, batch_size = 32, verbose = 0);
    plt.plot(history.history['val_loss'])
    
plt.xlabel('# epochs')
plt.ylabel('Training loss')

### Es posible llamar a early stopping como callback para detenr el procesos si se ve que no mejorará....

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping
early_stopping = EarlyStopping(monitor='val_loss', patience=2)

El argumento **patience = 2** impone al descenso de gradiente continuar 2 iteraciones (epochs) como tolerancia despues de una alza de la loss en la validación..... se supone que el loss debería disminuir a cada epoch

In [ ]:
model = Sequential()

model.add(Dense(H,input_dim=N, kernel_initializer = mon_init4))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1],kernel_initializer = mon_init4))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1],kernel_initializer = mon_init4))
model.add(Activation("softmax"))
sgd = SGD(learning_rate=0.1) 

model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'] )

history = model.fit(X_train_nor, Y_train,epochs = 100, batch_size = 32, verbose = 0, validation_split = 0.1, callbacks = [early_stopping])

In [ ]:
plt.plot(history.history['loss'], label = "train")
plt.plot(history.history['val_loss'], label = "valid")
plt.xlabel('# epochs')
plt.ylabel('Training loss')
plt.legend(loc = 'best');

> Encuentre la accuracy del último modelo ajustado (con early stopping) en el set 'X_test_nor'. Calcule el riesgo estimado en los datos test usando el método '.evaluate()'. Calcule finalmente la taza de malas clasificaciones en este set, es decir en porcenttaje cuántas veces nos equivocamos.

In [ ]:
y_predicted = np.argmax(model.predict(X_test_nor), axis=1)
print("Riesgo empírico y accuracy en el set test : ", model.evaluate(X_test_nor, Y_test))
mauvais_classement = 0
for i in range(len(y_predicted)) :
    if y_predicted[i] != y_test[i] :
        mauvais_classement += 1
mauvais_classement = mauvais_classement*100/len(y_predicted)
print("Porcentaje de equivocación en el set test : ", '%2d'%mauvais_classement, "%")

Se aconseja no detener el SGD una vez que el riesgo remonta, en particular cuando el **batch_size** es pequeño.
> Ilustre este fenómeno con los datos.

In [ ]:
early_stopping = EarlyStopping(monitor = 'val_loss', patience = 0)

model = Sequential()

model.add(Dense(H,input_dim=N, kernel_initializer = mon_init4))
model.add(Activation("tanh"))
model.add(Dense(H,kernel_initializer = mon_init4))
model.add(Activation("tanh"))
model.add(Dense(Y_train.shape[1],kernel_initializer = mon_init4))
model.add(Activation("softmax"))
sgd = SGD(learning_rate=0.1) 

model.compile(optimizer = sgd, loss = 'categorical_crossentropy', metrics = ['accuracy'] )

history = model.fit(X_train_nor, Y_train, epochs = 20, batch_size = 5, verbose = 0, validation_split = 0.1, callbacks = [early_stopping])

plt.plot(history.history['loss'], label = "train")
plt.plot(history.history['val_loss'], label = "valid")
plt.xlabel('# epochs')
plt.ylabel('Training loss')
plt.xlim(0,20)
plt.legend(loc = 'best');

> Vemos en la gráfica donde mostramos el trainning loss para varios epochs que, incluso si el erro augmenta un poco, no quiere decir que no vaya a disminuir después

## Aplicación : Datos MNIST

In [ ]:
# Aplicaci?n adicional: clasificaci?n de d?gitos con datos tipo MNIST
from sklearn.datasets import load_digits

try:
    from tensorflow.keras.datasets import mnist
    (x_train_m, y_train_m), (x_test_m, y_test_m) = mnist.load_data()
    x_train_m = x_train_m.reshape((-1, 28 * 28)).astype('float32') / 255.0
    x_test_m = x_test_m.reshape((-1, 28 * 28)).astype('float32') / 255.0
    y_train_cat = to_categorical(y_train_m, 10)
    y_test_cat = to_categorical(y_test_m, 10)

    model_mnist = Sequential()
    model_mnist.add(Dense(128, input_dim=28 * 28, activation='relu'))
    model_mnist.add(Dense(10, activation='softmax'))
    model_mnist.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model_mnist.fit(x_train_m[:12000], y_train_cat[:12000], epochs=2, batch_size=128, verbose=0)
    loss, acc = model_mnist.evaluate(x_test_m[:2000], y_test_cat[:2000], verbose=0)
    print('Exactitud MNIST (submuestra):', f'{acc:.4f}')
except Exception:
    # Respaldo sin descarga: digits de sklearn
    d = load_digits()
    Xd = d.data / 16.0
    yd = d.target
    Xtr, Xte, ytr, yte = train_test_split(Xd, yd, test_size=0.2, random_state=42)
    ytr_cat = to_categorical(ytr, 10)
    yte_cat = to_categorical(yte, 10)

    model_digits = Sequential()
    model_digits.add(Dense(64, input_dim=Xd.shape[1], activation='relu'))
    model_digits.add(Dense(10, activation='softmax'))
    model_digits.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    model_digits.fit(Xtr, ytr_cat, epochs=8, batch_size=32, verbose=0)
    loss, acc = model_digits.evaluate(Xte, yte_cat, verbose=0)
    print('Exactitud en digits (respaldo local):', f'{acc:.4f}')
